#  **Docling - 10K Report RAG** 


`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

---

## **PDF 기반 RAG 시스템 구축 (Docling & Langchain)**
    
- PDF를 `docling` 라이브러리로 파싱

- **Langchain**과 **ChromaDB**를 활용한 RAG 시스템 구축

- PDF 문서 기반 **지능형 질의응답** 시스템 구현 프로젝트

### 1. **문서 로드**

- 이전 수업에서 사용한 코드를 활용하여 PDF 문서를 로드합니다.


In [4]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.accelerator_options import AcceleratorDevice

class AdvancedDocProcessor:
    """고급 문서 처리기"""
    
    def __init__(self, enable_ocr=False, enable_table_structure=True):
        """
        Args:
            enable_ocr: OCR 기능 활성화 (스캔된 문서용)
            enable_table_structure: 테이블 구조 분석 활성화
        """
        
        # 파이프라인 옵션 설정
        pipeline_options = PdfPipelineOptions()  # PDF 변환을 위한 파이프라인 옵션 (기본값 사용)
        pipeline_options.do_ocr = enable_ocr   # OCR 활성화 여부
        pipeline_options.do_table_structure = enable_table_structure # 테이블 구조 분석 활성화 여부 
        
        # 테이블 구조 분석 세부 설정
        if enable_table_structure:
            pipeline_options.table_structure_options.do_cell_matching = True # 셀 매칭
        
        # CPU 사용 설정 (GPU가 없는 환경)
        pipeline_options.accelerator_options.device = AcceleratorDevice.CPU
        
        # DocumentConverter 초기화
        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options
                )
            }
        )
        
        print(f"🔧 처리기 초기화 완료:")
        print(f"   - OCR: {'활성화' if enable_ocr else '비활성화'}")
        print(f"   - 테이블 구조 분석: {'활성화' if enable_table_structure else '비활성화'}")
    
    def process_pdf(self, pdf_path):
        """PDF 처리"""
        try:
            print(f"📖 처리 시작: {pdf_path}")
            result = self.converter.convert(str(pdf_path))
            
            if result.status.name in ['SUCCESS', 'PARTIAL_SUCCESS']:
                print(f"✅ 처리 완료: {result.status.name}")
                return result.document
            else:
                print(f"❌ 처리 실패: {result.status}")
                return None
                
        except Exception as e:
            print(f"❌ 오류 발생: {e}")
            return None

In [6]:
# 고급 설정으로 처리
ocr_processor = AdvancedDocProcessor(
    enable_ocr=True,  # OCR 활성화
    enable_table_structure=True # 테이블 구조 분석 활성화
)

# PDF 처리 실행
pdf_path = "data/[별표 21] 주택관련 담보대출 등에 대한 위험관리 세부기준(제5-7조의 2관련)(보험업감독업무시행세칙).pdf"  # 변환할 PDF 문서 경로 
document = ocr_processor.process_pdf(pdf_path)

if document:
    # 결과 분석
    markdown = document.export_to_markdown()
    print(f"\n📊 분석 결과:")
    print(f"   - 문서명: {document.name}")
    print(f"   - 텍스트 길이: {len(markdown)}자")
    
    # 테이블이 있는지 확인
    if "| " in markdown or "|--" in markdown:
        print("   - 🔍 테이블 구조 감지됨")
    else:
        print("   - 📝 일반 텍스트 문서")

🔧 처리기 초기화 완료:
   - OCR: 활성화
   - 테이블 구조 분석: 활성화
📖 처리 시작: data/[별표 21] 주택관련 담보대출 등에 대한 위험관리 세부기준(제5-7조의 2관련)(보험업감독업무시행세칙).pdf
❌ 오류 발생: 
RTDetrImageProcessor requires the Torchvision library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.

RTDetrImageProcessor requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.



In [ ]:
# DoclingDocument의 텍스트 내용 확인
document.texts

In [ ]:
# 첫 번째 텍스트 아이템의 속성 확인
document.texts[0].model_dump()  

In [ ]:
# DoclingDocument의 테이블 구조 확인
document.tables

In [ ]:
# 첫 번째 테이블 아이템의 속성 확인
document.tables[0].model_dump() 

In [ ]:
# DoclingDocument의 텍스트와 테이블 요소를 순서대로 JSON으로 변환
from docling_core.types.doc import TextItem, TableItem
import pandas as pd

def convert_document_to_ordered_json(document):
    """문서 요소를 원본 순서대로 JSON 배열로 변환 (테이블은 마크다운+딕셔너리 형식)"""
    elements = []
    
    for item, level in document.iterate_items():
        if isinstance(item, TextItem):
            elements.append({
                "type": "text",
                "content": item.text.replace("\t", " ") or "",
                "page": item.prov[0].page_no if item.prov else None,
                "label": item.label.value if item.label else None,
                "level": level,
                "element": item  # 원본 요소 추가
            })
        elif isinstance(item, TableItem):
            table_content = {}
            
            try:
                # DataFrame으로 변환
                df = item.export_to_dataframe()

                # 마크다운 형식으로 변환
                markdown_content = df.to_markdown(index=False)  # 인덱스 제외

                # 딕셔너리 형식으로 변환 (여러 옵션 제공)
                dict_content = df.to_dict('records')     # 각 행을 딕셔너리로

                table_content = {
                    "markdown": markdown_content,
                    "data": dict_content,
                    "status": "success"
                }
                
            except Exception as e:
                # DataFrame 변환이 실패한 경우 대안 시도
                try:
                    html_content = item.export_to_html()
                    table_content = {
                        "html": html_content,
                        "status": "html_fallback",
                        "error": str(e)
                    }
                except Exception as e2:
                    table_content = {
                        "status": "failed",
                        "error": f"DataFrame 변환 실패: {str(e)}, HTML 변환 실패: {str(e2)}"
                    }
            
            elements.append({
                "type": "table",
                "content": table_content,
                "page": item.prov[0].page_no if item.prov else None,
                "level": level,
                "element": item  # 원본 요소 추가
            })
    
    return elements


# 문서 요소를 원본 순서대로 JSON 배열로 변환
ordered_json = convert_document_to_ordered_json(document)
print(f"문서 요소를 원본 순서대로 JSON 배열로 변환 완료. 총 {len(ordered_json)}개 요소.")

In [ ]:
ordered_json[:10]

In [ ]:
# 표(table) 타입인 요소들의 인덱스 목록
table_indices = [i for i, item in enumerate(ordered_json) if item['type'] == 'table']
print(f"총 {len(table_indices)}개 표 발견")
print(table_indices)


In [ ]:
# 표 요소의 내용 확인 (표)
print(ordered_json[362]['content']['markdown']  )

In [ ]:
# 표 요소의 내용 확인 (표)
pd.DataFrame(ordered_json[362]['content']['data'])

In [ ]:
# pickle로 저장
from pathlib import Path
import pickle

save_folder = Path("data/docling_output")
pickle_path = save_folder / "tsla_analysis_ocr.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(ordered_json, f)
print(f"pickle 파일로 저장됨: {pickle_path}")

In [ ]:
# pickle로 저장한 데이터 로드
from pathlib import Path
import pickle

save_folder = Path("data/docling_output")
pickle_path = save_folder / "tsla_analysis_ocr.pkl"
with open(pickle_path, "rb") as f:
    elements = pickle.load(f)
print(f"pickle 파일로 로드됨: {pickle_path}")

In [ ]:
len(elements)  # 데이터의 길이 확인

In [ ]:
elements[:10]

In [ ]:
# 363번째 요소의 내용 확인 (표)
print(elements[362]['content']['markdown'])

### 2. **LangChain 문서 변환**

- **LangChain**을 사용하여 문서를 **Document** 객체로 변환

In [ ]:
from langchain_core.documents import Document

def quick_convert(ordered_json):
    documents = []
    for item in ordered_json:
        content = ""
        if item.get('type') == 'text':
            content = item.get('content', '')
        elif item.get('type') == 'table' and isinstance(item.get('content'), dict):
            content = item['content'].get('markdown', '[TABLE]')
        
        if content.strip():
            documents.append(Document(
                page_content=content,
                metadata={'page': item.get('page', 0), 'type': item.get('type')}
            ))
    return documents

# 바로 사용
docs = quick_convert(elements)
print(f"✅ 변환 완료: {len(docs)}개 LangChain Document")

In [ ]:
docs[350:380]  # 변환된 문서의 일부 확인

In [ ]:
def convert_by_page(ordered_json):
    """페이지별로 합쳐서 LangChain Document로 변환"""
    # 페이지별로 그룹화
    pages = {}
    for item in ordered_json:
        page_num = item.get('page', None)
        if page_num is not None:
            if page_num not in pages:
                pages[page_num] = []
            pages[page_num].append(item)
    
    documents = []
    for page_num, items in sorted(pages.items()):
        # 페이지 내용 합치기
        contents = []
        for item in items:
            if item.get('type') == 'text' and item.get('content'):
                # "Table of Contents"와 같은 특정 텍스트는 제외

                for exclude_text in ["Table of Contents", "TABLE OF CONTENTS"]:
                    if exclude_text in item['content']:
                        cleaned_content = item['content'].replace(exclude_text, '').strip()
                        contents.append(cleaned_content)
                        break
                else:
                    contents.append(item['content'].strip())

            elif item.get('type') == 'table':
                table_content = item.get('content', {})
                if isinstance(table_content, dict) and 'markdown' in table_content:
                    contents.append(table_content['markdown'])
        
        if contents:
            doc = Document(
                page_content='\n\n'.join(contents).strip(),
                metadata={
                    'page': page_num,
                    'elements_count': len(items),                }
            )
            documents.append(doc)

        else:
            # 빈 페이지는 추가하지 않음
            print(f"⚠️ 페이지 {page_num}은 내용이 없습니다. 건너뜁니다.")
    
    return documents

# 페이지별로 변환
docs_by_page = convert_by_page(elements)
print(f"✅ 페이지별 변환 완료: {len(docs_by_page)}개 LangChain Document")

In [ ]:
print(docs_by_page[2].page_content)

In [ ]:
pprint(docs_by_page[2].metadata)

### 3. **문서 계층화**

- **문서 계층화**는 대규모 문서를 의미 있는 **작은 단위로 분할**하는 기술

- 3페이지(page_number) **목차 항목**을 기준으로 그룹화 

`(1) 목차 항목 추출`

In [ ]:
# 3페이지(page_number) 목차 항목을 기준으로 그룹화 

toc_items = docs_by_page[2].page_content
print(toc_items)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
import json

# Pydantic 모델 정의
class Item(BaseModel):
    """목차 항목 모델"""
    number: str = Field(description="목차 항목 번호 (예: 1, 1A, 1B, 2, 3 등)")
    title: str = Field(description="목차 항목 제목")
    page: int = Field(description="목차 항목이 위치한 페이지 번호")

class Section(BaseModel):
    """목차 섹션 모델"""
    section: str = Field(description="목차 항목 그룹 (예: PART I, PART II 등)")
    items: List[Item] = Field(description="목차 항목 리스트")

class TOCStructure(BaseModel):
    """전체 목차 구조"""
    sections: List[Section] = Field(description="목차 섹션 리스트")

# LangChain 프롬프트 템플릿 정의
toc_extraction_prompt = PromptTemplate(
            input_variables=["table_text"],
            template="""
You are an expert document analyzer specializing in extracting table of contents information from financial documents.

Your task is to analyze the following table of contents text and extract structured information about sections and items.

Key requirements:
1. Identify PART sections (e.g., "PART I", "PART II", "PART III", "PART IV")
2. Extract Item numbers (e.g., "Item 1", "Item 1A", "Item 2", etc.)
3. Extract the title/description for each item
4. Extract the page number for each item
5. Group items under their corresponding PART sections

Important notes:
- Some pages may have multiple items starting on the same page
- Item numbers can include letters (e.g., "1A", "7A", "9A")
- Ignore table headers and formatting lines
- Focus only on actual content items

Table of Contents Text:
{table_text}

Please extract this information and structure it properly with sections and their corresponding items.
"""
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm_structured = llm.with_structured_output(TOCStructure)

toc_extractor = toc_extraction_prompt | llm_structured

toc_sections = toc_extractor.invoke(docs_by_page[2].page_content) 
toc_sections

`(2) 섹션별로 그룹화`

In [ ]:
# 섹션별로 그룹화된 목차 항목 출력
toc_by_section = {}

for section in toc_sections.sections:
    toc_by_section[section.section] = []
    for item in section.items:
        toc_by_section[section.section].append({
            'number': item.number,
            'title': item.title,
            'page': item.page
        })

# 출력 예시
for section, items in toc_by_section.items():
    print(f"Section: {section}")
    for item in items:
        print(f"  {item['number']}: {item['title']} (Page {item['page']})")

In [ ]:
toc_by_section['PART I']

In [ ]:
item_list = toc_by_section.values()
item_list = [item for sublist in item_list for item in sublist] 

item_list

In [ ]:
import re
from typing import List, Dict, Tuple
from langchain_core.documents import Document

def get_item_header(item_number: str, item_title: str) -> List[str]:
    """가능한 목차 항목 제목 변형을 반환하는 함수"""
    variations = [
        f"{item_number}. {item_title}",
        f"{item_number.upper()}. {item_title.upper()}",
        f"ITEM {item_number.upper()}. {item_title.upper()}",
    ]
    return variations

def find_all_section_positions(page_content: str, item_list: List[Dict]) -> List[Tuple[int, Dict]]:
    """
    페이지 내용에서 모든 섹션의 시작 위치를 찾아 반환
    
    Returns:
        List of tuples: (start_position, item_dict)
    """
    section_positions = []
    
    for item in item_list:
        item_headers = get_item_header(item['number'], item['title'])
        
        for header in item_headers:
            # 정규표현식을 사용하여 더 정확한 매칭
            # 헤더가 줄의 시작 부분에 있거나 앞에 공백만 있는 경우를 찾음
            pattern = r'(?:^|\n)\s*' + re.escape(header)
            matches = list(re.finditer(pattern, page_content, re.MULTILINE | re.IGNORECASE))
            
            for match in matches:
                section_positions.append((match.start(), item))
                break  # 같은 item에 대해 하나만 찾으면 됨
            
            if matches:  # 이미 찾았으면 다른 변형은 확인하지 않음
                break
    
    # 위치 순으로 정렬
    section_positions.sort(key=lambda x: x[0])
    return section_positions

def split_document_by_sections(doc: Document, item_list: List[Dict]) -> List[Document]:
    """
    하나의 문서를 여러 섹션으로 분할
    
    Returns:
        분할된 문서들의 리스트
    """
    page_content = doc.page_content
    section_positions = find_all_section_positions(page_content, item_list)
    
    if not section_positions:
        # 섹션을 찾을 수 없으면 원본 문서 그대로 반환
        return [Document(
            page_content=doc.page_content,
            metadata={**doc.metadata, "section": "Unknown"}
        )]
    
    split_docs = []
    
    for i, (start_pos, item) in enumerate(section_positions):
        # 현재 섹션의 끝 위치 결정
        if i < len(section_positions) - 1:
            end_pos = section_positions[i + 1][0]
        else:
            end_pos = len(page_content)
        
        # 섹션 내용 추출
        section_content = page_content[start_pos:end_pos].strip()
        
        if section_content:  # 빈 내용이 아닌 경우만 추가
            split_doc = Document(
                page_content=section_content,
                metadata={
                    **doc.metadata,
                    "section": item['title'],
                    "section_number": item['number'],
                    "section_start_char": start_pos,
                    "section_end_char": end_pos
                }
            )
            split_docs.append(split_doc)
    
    return split_docs


def process_documents_with_toc(docs_by_page: List[Document], item_list: List[Dict]) -> List[Document]:
    """
    목차 기반으로 문서들을 처리하여 섹션별로 분할 
    각 섹션에 속하는 모든 문서에 일관된 섹션 정보 추가
    """
    # 1단계: 각 섹션의 시작 위치 찾기
    section_boundaries = []  # [(page_num, item_info), ...]
    
    for i, doc in enumerate(docs_by_page):
        page_num = doc.metadata.get('page', i)
        section_positions = find_all_section_positions(doc.page_content, item_list)
        
        for pos, item in section_positions:
            section_boundaries.append({
                'page_index': i,
                'page_num': page_num,
                'position': pos,
                'item': item,
                'doc': doc
            })
    
    # 페이지 번호와 위치로 정렬
    section_boundaries.sort(key=lambda x: (x['page_index'], x['position']))
    
    # 2단계: 섹션별로 문서 처리
    toc_based_docs = []
    
    for doc_idx, doc in enumerate(docs_by_page):
        page_num = doc.metadata.get('page', doc_idx)
        
        # 현재 페이지의 섹션 찾기
        page_sections = [sb for sb in section_boundaries if sb['page_index'] == doc_idx]
        
        if page_sections:
            # 페이지에 섹션이 있는 경우 - 분할 처리
            split_docs = split_document_by_sections(doc, item_list)
            toc_based_docs.extend(split_docs)
        else:
            # 페이지에 섹션이 없는 경우 - 이전 섹션 찾기
            prev_sections = [sb for sb in section_boundaries if sb['page_index'] < doc_idx]
            
            if prev_sections:
                # 가장 최근 섹션 정보 사용
                last_section = prev_sections[-1]['item']
                doc_copy = Document(
                    page_content=doc.page_content,
                    metadata={
                        **doc.metadata,
                        "section": last_section['title'],
                        "section_number": last_section['number'],
                        "section_continued": True  # 이전 섹션의 연속임을 표시
                    }
                )
            else:
                # 아직 섹션이 시작되지 않은 경우
                doc_copy = Document(
                    page_content=doc.page_content,
                    metadata={
                        **doc.metadata,
                        "section": "Preamble",  # "Unknown" 대신 더 명확한 이름
                        "section_number": "0"
                    }
                )
            toc_based_docs.append(doc_copy)
    
    return toc_based_docs

# 섹션별 통계를 더 자세히 보여주는 함수
def analyze_sections(toc_based_docs: List[Document]) -> Dict:
    """
    섹션별 문서 분석 및 통계 생성
    """
    section_stats = {}
    
    for doc in toc_based_docs:
        section = doc.metadata.get("section", "Unknown")
        section_num = doc.metadata.get("section_number", "")
        
        if section not in section_stats:
            section_stats[section] = {
                "section_number": section_num,
                "doc_count": 0,
                "pages": set(),
                "total_chars": 0,
                "has_splits": False,
                "continued_pages": 0
            }
        
        stats = section_stats[section]
        stats["doc_count"] += 1
        stats["pages"].add(doc.metadata.get("page", "N/A"))
        stats["total_chars"] += len(doc.page_content)
        
        if doc.metadata.get("section_continued", False):
            stats["continued_pages"] += 1
        if doc.metadata.get("section_start_char") is not None:
            stats["has_splits"] = True
    
    # 페이지 집합을 정렬된 리스트로 변환
    for section in section_stats:
        pages = section_stats[section]["pages"]
        section_stats[section]["pages"] = sorted([p for p in pages if p != "N/A"])
        section_stats[section]["page_range"] = f"{min(pages)} - {max(pages)}" if pages else "N/A"
    
    return section_stats

# 문서 처리
toc_based_docs = process_documents_with_toc(docs_by_page, item_list)

# 특정 섹션의 모든 문서 확인
business_docs = [
    doc for doc in toc_based_docs 
    if doc.metadata.get("section") == "Business"
]

# 모든 문서가 동일한 section_number를 가짐
for doc in business_docs:
    print(f"Page {doc.metadata['page']}: {doc.metadata['section_number']}")
    # 출력: Page 5: Item 1, Page 6: Item 1, Page 7: Item 1, ...

In [ ]:
# 특정 섹션의 문서 메타데이터 확인
toc_based_docs[4].metadata

In [ ]:
# 특정 섹션의 문서 메타데이터 확인
toc_based_docs[5].metadata

In [ ]:
# 문서 저장
with open("data/docling_tesla_10k_toc_based_docs.pkl", "wb") as f:
    pickle.dump(toc_based_docs, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
import pickle
from pathlib import Path

pickle_path = Path("data/docling_tesla_10k_toc_based_docs.pkl")

# 파일 존재 확인
if not pickle_path.exists():
    print(f"❌ 파일이 존재하지 않습니다: {pickle_path}")
    print("먼저 이전 셀을 실행하여 문서를 생성하고 저장하세요.")
else:
    with open(pickle_path, "rb") as f:
        toc_based_docs = pickle.load(f)
    print(f"✅ 문서 로드 완료: {len(toc_based_docs)}개")

In [ ]:
toc_based_docs

`(3) 섹션별로 결합`

- 보고서 섹션별로 문서 객체를 결합하여 재구조화 

In [ ]:
from collections import defaultdict
from langchain_core.documents import Document

# 섹션별로 문서 그룹화
section_groups = defaultdict(list)
for doc in toc_based_docs:
    section = doc.metadata.get('section', 'Unknown')
    section_groups[section].append(doc)

# "Preamble"과 "Unknown" 섹션 제외
EXCLUDE_SECTIONS = ["Preamble", "Unknown"]

# 섹션별로 문서 결합 ([PAGE:N] 마커로 페이지 경계 표시)
# [PAGE:N] 마커는 청크 분할 후에도 페이지 번호를 추적하기 위한 장치
section_docs_combined = {}

for section_name, docs in section_groups.items():
    if section_name in EXCLUDE_SECTIONS:
        continue
    if not docs:
        continue

    # 페이지 순서로 정렬 후 [PAGE:N] 마커와 함께 결합
    page_contents = []
    for doc in sorted(docs, key=lambda d: d.metadata.get('page', 0)):
        page_num = doc.metadata.get('page', 0)
        page_contents.append(f"[PAGE:{page_num}]\n{doc.page_content}")

    combined_content = "\n\n".join(page_contents)

    # 페이지 범위 계산
    pages = [doc.metadata.get('page', 0) for doc in docs]
    start_page = min(pages)
    end_page = max(pages)
    section_number = docs[0].metadata.get('section_number', '')

    section_docs_combined[section_name] = Document(
        page_content=combined_content,
        metadata={
            'section': section_name,
            'section_number': section_number,
            'start_page': start_page,
            'end_page': end_page,
            'page_count': len(pages),
        }
    )

print(f"결합된 섹션 수: {len(section_docs_combined)}")
print()
print("섹션별 페이지 범위:")
for section, doc in section_docs_combined.items():
    m = doc.metadata
    print(f"  {section}: p.{m['start_page']}-{m['end_page']} ({m['page_count']}페이지)")

In [ ]:
# 문서 저장
with open("data/docling_tesla_10k_sections.pkl", "wb") as f:
    pickle.dump(section_docs_combined, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
import pickle
from pathlib import Path

pickle_path = Path("data/docling_tesla_10k_sections.pkl")

# 파일 존재 확인
if not pickle_path.exists():
    print(f"❌ 파일이 존재하지 않습니다: {pickle_path}")
    print("먼저 이전 셀을 실행하여 문서를 생성하고 저장하세요.")
else:
    with open(pickle_path, "rb") as f:
        section_docs_combined = pickle.load(f)
    print(f"✅ 문서 로드 완료: {len(section_docs_combined)}개")

In [ ]:
# 첫 번째 문서의 내용 출력
print(f"{section_docs_combined['Business'].page_content}\n")

In [ ]:
# 첫 번째 섹션의 메타데이터 확인
pprint(section_docs_combined['Business'].metadata)

### 4. **Parent Document Retriever**

- 문서 검색 시 **작은 단위 분할**과 **문맥 유지** 사이의 균형이 중요함

- 작은 단위로 분할하면 **임베딩의 정확도**가 높아지나 문맥이 손실될 수 있음

- **ParentDocumentRetriever**는 작은 청크로 저장하고 검색 시 상위 문서를 반환하여 두 가지 목표를 달성함

- 효과적인 문서 검색을 위해 청크 크기와 문맥 보존 사이의 최적점을 찾는 것이 핵심

In [ ]:
import matplotlib.pyplot as plt

# 각 섹션별로 문서 길이 확인
section_lengths = {section: len(doc.page_content) for section, doc in section_docs_combined.items()}

# 길이 순서로 정렬
section_lengths = dict(sorted(section_lengths.items(), key=lambda item: item[1]))

# 섹션별 문서 길이 시각화
plt.barh(section_lengths.keys(), section_lengths.values())
plt.xticks(rotation=90)
plt.xlabel("Section")
plt.ylabel("Length")
plt.title("Document Lengths by Section")
plt.show()

In [ ]:
import tiktoken

# tiktoken tokenizer 초기화
tokenizer = tiktoken.get_encoding("cl100k_base")

# 각 섹션별로 토큰 수 확인
section_tokens = {section: len(tokenizer.encode(doc.page_content)) for section, doc in section_docs_combined.items()}

# 길이 순서로 정렬
section_tokens = dict(sorted(section_tokens.items(), key=lambda item: item[1]))

# 섹션별 토큰 수 시각화
plt.barh(section_tokens.keys(), section_tokens.values())
plt.xticks(rotation=90)
plt.xlabel("Section")
plt.ylabel("Tokens")
plt.title("Document Tokens by Section")
plt.show()

In [ ]:
# 첫 번째 섹션의 메타데이터 확인
pprint(section_docs_combined['Business'].metadata)

In [ ]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def extract_page_from_chunk(chunk_content: str, fallback_page: int) -> int:
    """청크 내용에서 [PAGE:N] 마커를 찾아 페이지 번호 추출"""
    matches = re.findall(r'\[PAGE:(\d+)\]', chunk_content)
    if matches:
        return int(matches[0])
    return fallback_page


def clean_page_markers(content: str) -> str:
    """청크 내용에서 [PAGE:N] 마커 제거"""
    return re.sub(r'\[PAGE:\d+\]\n?', '', content).strip()


# 토큰 기반 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "? ", "! ", ". "]
)

# 섹션별 분할 ([PAGE:N] 마커에서 페이지 정보 추출)
section_docs_split = {}

for section_name, doc in section_docs_combined.items():
    start_page = doc.metadata['start_page']

    # 텍스트 분할
    split_docs = text_splitter.split_documents([doc])

    # 마지막으로 감지된 페이지 추적 (순차 처리)
    last_seen_page = start_page

    for i, split_doc in enumerate(split_docs):
        # [PAGE:N] 마커에서 페이지 번호 추출
        page = extract_page_from_chunk(split_doc.page_content, last_seen_page)
        last_seen_page = page

        # 마커 제거 후 메타데이터 설정
        split_doc.page_content = clean_page_markers(split_doc.page_content)
        split_doc.metadata = {
            'section': section_name,
            'page': page,
            'order': i + 1,
        }

    section_docs_split[section_name] = split_docs

# 결과 확인
total_chunks = sum(len(docs) for docs in section_docs_split.values())
print(f"총 청크 수: {total_chunks}")
print()
print("섹션별 청크 수:")
for section, chunks in section_docs_split.items():
    pages = sorted(set(c.metadata['page'] for c in chunks))
    page_range = f"p.{min(pages)}-{max(pages)}" if pages else "N/A"
    print(f"  {section}: {len(chunks)}개 ({page_range}, {len(pages)}개 페이지)")

In [ ]:
section_docs_split['Business'][0].metadata 

In [ ]:
# 문서 객체를 pickle로 저장
with open("data/docling_tesla_10k_sections_split.pkl", "wb") as f:
    pickle.dump(section_docs_split, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
from pathlib import Path

pickle_path = Path("data/docling_tesla_10k_sections_split.pkl")

# 파일 존재 확인
if not pickle_path.exists():
    print(f"❌ 파일이 존재하지 않습니다: {pickle_path}")
    print("먼저 이전 셀을 실행하여 문서를 생성하고 저장하세요.")
else:
    with open(pickle_path, "rb") as f:
        section_docs_split = pickle.load(f)
    print(f"✅ 문서 로드 완료: {len([doc for docs in section_docs_split.values() for doc in docs])}개")

In [ ]:
section_docs_split

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os
import pickle

# 문서 저장소 경로 설정
storage_path = "./docling_document_store"
os.makedirs(storage_path, exist_ok=True)

# 부모 문서 저장소 클래스 정의
class PickleFileStore(LocalFileStore):
    def mget(self, keys):
        """Get the values for the given keys."""
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        """Set the values for the given key-value pairs."""
        serialized_pairs = [
            (k, pickle.dumps(v)) for k, v in key_value_pairs
        ]
        super().mset(serialized_pairs)

# 부모 문서 저장소 초기화
store = PickleFileStore(storage_path)

# 부모 문서용 텍스트 스플리터 (섹션 단위로 큰 청크)
parent_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000,
    chunk_overlap=400,
    separators=["\n\n", "\n", r"(?<=[.!?])\s+"] # 문장 마침표 정규식 추가
)

# 자식 문서용 텍스트 스플리터 (검색용 작은 청크)
child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", r"(?<=[.!?])\s+"]
)

# 벡터 스토어 초기화
vectorstore = Chroma(
    collection_name="docling_tesla_10k_sections",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
retriever.add_documents(
    [
        doc
        for docs in section_docs_split.values()
        for doc in docs
    ]
)

In [ ]:
# 저장된 문서 수 확인 
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")

# 로컬 저장소 문서 수 확인
all_keys = list(store.yield_keys())
print(f"Total documents in store: {len(all_keys)}")

In [ ]:
# Test 검색
query = "What are the main risk factors?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents\n")
for doc in retrieved_docs:
    section = doc.metadata.get('section', 'N/A')
    page = doc.metadata.get('page', 'N/A')
    order = doc.metadata.get('order', 'N/A')
    print(f"[Section: {section} | Page: {page} | Order: {order}]")
    print(doc.page_content[:500])
    print()
    print("=" * 100)

In [ ]:
# 벡터 스토어 로드
vectorstore = Chroma(
    collection_name="docling_tesla_10k_sections",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 로컬 파일 저장소 로드
storage_path = "./docling_document_store"
store = PickleFileStore(storage_path)


# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Test 검색
query = "What are the main risk factors?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print()
    print("=" * 100)

In [ ]:
# 다른 쿼리
query = "Where is Tesla's headquarters located?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents\n")

for doc in retrieved_docs:
    section = doc.metadata.get('section', 'N/A')
    page = doc.metadata.get('page', 'N/A')
    order = doc.metadata.get('order', 'N/A')
    print(f"[Section: {section} | Page: {page} | Order: {order}]")
    print(doc.page_content[:500])
    print()
    print("=" * 100)

### 5. **다중 벡터 기반 검색(Multi Vector Retrieval)**

- **Multi-Vector Retriever**는 문서를 **여러 벡터**로 분할하여 저장하고 검색하는 시스템
    1. 벡터 저장소(Vectorstore): 임베딩 저장
    2. 문서 저장소(Docstore): 원본 문서 저장

- 예제: **요약 기반 검색 시스템** 구현

    - 각 섹션의 간결한 요약문을 생성하여 벡터 저장소에 보관하고 유사도 검색에 활용함
    - **영구 저장소**는 PickleFileStore와 Chroma를 사용하여 문서와 벡터 데이터를 세션 간 유지함
    - **검색 과정**은 요약문으로 의미론적 매칭을 수행하고 전체 원본 문서를 반환하여 완전한 문맥을 제공함
    - **RAG 파이프라인**은 ChatGPT를 활용하여 요약 생성과 최종 답변을 처리하며 명확한 응답 형식을 제공함

In [ ]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_classic.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
import pickle

# 로컬 파일 저장소 클래스 정의
class PickleFileStore(LocalFileStore):
    def mget(self, keys):
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        serialized_pairs = [(k, pickle.dumps(v)) for k, v in key_value_pairs]
        super().mset(serialized_pairs)

# 저장소 디렉토리 생성
os.makedirs("./summary_store", exist_ok=True)
os.makedirs("./chroma_db", exist_ok=True)

# 로컬 파일 저장소 초기화
doc_store = PickleFileStore("./summary_store")
vectorstore = Chroma(
    collection_name="tesla_10k_summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# MultiVectorRetriever 설정
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id"   # 문서 ID 키 설정
)

# 각 문서에 대한 요약 생성
def generate_summary(text):
    summary_prompt = ChatPromptTemplate.from_template(
        "Summarize the following 10-K report section in 2-3 sentences:\n\n{text}"
    )
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    summary_chain = summary_prompt | model | StrOutputParser()
    return summary_chain.invoke({"text": text})

# 요약된 문서 저장
summary_docs = []
original_docs = []
doc_ids = []

# 각 섹션별로 문서 결합
to_index_docs = [
        doc
        for docs in section_docs_split.values()
        for doc in docs
]

# 각 문서에 대한 요약 생성
for i, doc in enumerate(to_index_docs):
    doc_id = f"doc_{i}"
    summary = generate_summary(doc.page_content)

    # 요약 문서 생성 (페이지/순서 메타데이터 포함)
    summary_doc = Document(
        page_content=summary,
        metadata={
            "doc_id": doc_id,
            "section": doc.metadata["section"],
            "page": doc.metadata.get("page", 0),
            "order": doc.metadata.get("order", 1),
        }
    )
    summary_docs.append(summary_doc)

    # 원본 문서와 ID 저장
    original_docs.append(doc)
    doc_ids.append(doc_id)

# 벡터 스토어에 요약 문서 추가
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, original_docs)))

In [ ]:
# 로컬 파일 저장소 로드
doc_store = PickleFileStore("./summary_store")
vectorstore = Chroma(
    collection_name="tesla_10k_summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 문서 개수 확인
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")
all_keys = list(doc_store.yield_keys())
print(f"Total documents in store: {len(all_keys)}")

In [ ]:
# RAG 체인 생성
rag_prompt = ChatPromptTemplate.from_template("""
Answer the following question based on the provided context from a 10-K report.
If you cannot find the answer in the context, say "제공된 문서에서 답을 찾을 수 없습니다."

<Context>
{context}
</Context>

<Question>
{question}
</Question>

<Answer>""")

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

# 질문 생성
question = "What are the main risk factors?"
answer = chain.invoke(question)
# answer = chain.invoke(question, config={"callbacks": [langfuse_handler]})
print(f"\nQuestion: {question}")
print(f"Answer: {answer}")

---

## **실습 문제**

### **문제 1: 다른 섹션 질문 테스트**

다음 질문들로 RAG 시스템을 테스트해보세요:
- "Where is Tesla's headquarters located?"
- "What is Tesla's total revenue in 2024?"
- "Who are Tesla's competitors?"

각 질문에 대해 답변과 함께 **출처(섹션, 페이지)** 정보를 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>정답 확인</summary>

```python
questions = [
    "Where is Tesla's headquarters located?",
    "What is Tesla's total revenue in 2024?",
    "Who are Tesla's competitors?"
]

for question in questions:
    answer = chain.invoke(question, config={"callbacks": [langfuse_handler]})
    print(f"\nQuestion: {question}")
    print(f"Answer: {answer}")

    # 검색된 문서의 출처(섹션, 페이지) 확인
    source_docs = retriever.invoke(question)
    sources = set()
    for doc in source_docs:
        section = doc.metadata.get("section", "N/A")
        page = doc.metadata.get("page", "N/A")
        sources.add(f"{section} (p.{page})")
    print(f"Sources: {', '.join(sorted(sources))}")
    print("=" * 80)
```

</details>

---

### **문제 2: 검색 k값 비교**

`MultiVectorRetriever`의 `search_kwargs`에서 `k` 값을 변경하여 검색 결과를 비교해보세요:
- k=3과 k=5로 각각 `MultiVectorRetriever`를 설정
- 동일한 질문에 대해 두 retriever로 RAG 체인을 구성하고 답변 비교
- 더 많은 문서를 참조할 때 답변이 어떻게 달라지는지 관찰

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>정답 확인</summary>

```python
from langchain_classic.retrievers import MultiVectorRetriever

# k=3으로 설정
retriever_k3 = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id",
    search_kwargs={"k": 3}
)

# k=5로 설정
retriever_k5 = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id",
    search_kwargs={"k": 5}
)

question = "What are the main risk factors?"

# k=3 결과
chain_k3 = (
    {"context": retriever_k3, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)
answer_k3 = chain_k3.invoke(question, config={"callbacks": [langfuse_handler]})

# k=5 결과
chain_k5 = (
    {"context": retriever_k5, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)
answer_k5 = chain_k5.invoke(question, config={"callbacks": [langfuse_handler]})

print("=== k=3 결과 ===")
print(answer_k3)
print("\n=== k=5 결과 ===")
print(answer_k5)
```

</details>

---

### **문제 3: 요약 프롬프트 수정**

`generate_summary` 함수의 프롬프트를 수정하여 더 상세한 요약을 생성해보세요:
- 3-5 문장으로 요약 길이 조정
- 특정 정보(숫자, 날짜 등)를 강조하도록 프롬프트 수정
- 원본 요약과 상세 요약 결과를 비교

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>정답 확인</summary>

```python
def generate_detailed_summary(text):
    """3-5 문장으로 상세한 요약을 생성하며 숫자와 날짜를 강조"""
    detailed_summary_prompt = ChatPromptTemplate.from_template(
        """Summarize the following 10-K report section in 3-5 sentences.

        Focus on:
        - Key financial figures (revenue, profits, costs, etc.)
        - Important dates and time periods
        - Significant numbers and percentages
        - Major events or changes

        Text:
        {text}
        """
    )
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    summary_chain = detailed_summary_prompt | model | StrOutputParser()
    return summary_chain.invoke({"text": text})

# 첫 번째 문서로 테스트
test_doc = to_index_docs[0]

print("=== 원본 요약 ===")
print(generate_summary(test_doc.page_content))
print("\n=== 상세 요약 ===")
print(generate_detailed_summary(test_doc.page_content))
```

</details>